# Excel Intelligence & Data Engineering Consultant
This notebook processes Excel files from an `input` folder, analyzes every sheet using AI, and generates a detailed audit CSV in an `Output` folder.

In [ ]:
import os
import pandas as pd
from pathlib import Path

class ExcelDataConsultant:
    """Handles local discovery and automated engineering analysis of Excel and CSV data."""

    def __init__(self, input_dir='input', output_dir='Output'):
        self.input_path = Path(input_dir)
        self.output_path = Path(output_dir)
        self.output_path.mkdir(exist_ok=True)
        self.results = []

    def local_data_analysis(self, file_name, sheet_name, df):
        """Performs a programmatic audit of the data without external AI."""
        null_counts = df.isnull().sum()
        total_nulls = null_counts.sum()
        cols = df.columns.tolist()
        
        needs_cleansing = "Yes" if total_nulls > 0 else "No"
        cleansing_cause = []
        if total_nulls > 0: cleansing_cause.append(f"{total_nulls} null values found")
        if df.duplicated().any(): cleansing_cause.append("Duplicate rows detected")
        
        domain = "General/Unknown"
        keywords = {
            "Healthcare": ["patient", "clinic", "diagnosis", "medical"],
            "Finance": ["amount", "transaction", "balance", "revenue"],
            "Government": ["census", "district", "citizen", "policy"]
        }
        for d, keys in keywords.items():
            if any(k in " ".join(cols).lower() for k in keys):
                domain = d
                break

        return {
            "summary": f"Dataset with {len(df)} rows and {len(cols)} columns.",
            "guidance": "Automated profiling: check data types and null distribution.",
            "engineering_comments": f"Cleansing required: {needs_cleansing}",
            "cleansing_cause": "; ".join(cleansing_cause) if cleansing_cause else "Data appears clean",
            "suggested_domain": domain
        }

    def process_all_files(self):
        """Walks through the input directory and processes files."""
        print(f"Stage 1: Searching for files in '{self.input_path.absolute()}'...")
        files = []
        for ext in ['*.xlsx', '*.xls', '*.csv']:
            files.extend(list(self.input_path.glob(ext)))

        if not files:
            print("Warning: No files found. Please ensure files are inside the 'input' folder.")
            return

        print(f"Stage 2: Found {len(files)} file(s). Starting analysis...")
        for file in files:
            print(f"  -> Processing: {file.name}")
            try:
                if file.suffix.lower() == '.csv':
                    df = pd.read_csv(file)
                    self._run_analysis(file.name, "CSV_Main_Sheet", df)
                else:
                    xl = pd.ExcelFile(file)
                    for sheet in xl.sheet_names:
                        df = pd.read_excel(file, sheet_name=sheet)
                        self._run_analysis(file.name, sheet, df)
            except Exception as e:
                print(f"  !! Error processing {file.name}: {e}")

    def _run_analysis(self, file_name, sheet_name, df):
        insights = self.local_data_analysis(file_name, sheet_name, df)
        self.results.append({
            "file_name": file_name,
            "sheet_name": sheet_name,
            "data_profile": f"Rows: {len(df)}, Cols: {len(df.columns)}",
            **insights
        })

    def save_report(self):
        print("Stage 3: Generating final report...")
        if not self.results:
            print("Aborted: No results found to save.")
            return
        report_df = pd.DataFrame(self.results)
        output_file = self.output_path / "data_analysis_report.csv"
        report_df.to_csv(output_file, index=False)
        print(f"Stage 4: Success! Report saved to: {output_file}")

# --- AUTOMATED EXECUTION ---
print("--- Starting Excel/CSV Data Audit ---")
# Create input directory if it doesn't exist for the user
Path('input').mkdir(exist_ok=True)

consultant = ExcelDataConsultant()
consultant.process_all_files()
consultant.save_report()
print("--- Audit Complete ---")

### Instructions to Run:
1. Ensure you have an `input` folder in your current directory with your Excel files.
2. Run the code above to generate the `excel_analysis_report.csv` in the `Output` folder.